# Exploratory Data Analysis

The objective of this notebook is to understand the structure of the dataset, identify fraud-related patterns, investigate missing-value behavior, and generate hypotheses for feature engineering.

In [ ]:
import pandas as pd

df = pd.read_csv("../data/raw/DataSet.csv")

print(df.shape)

df.head()

df.info()

## Dataset Overview and Missing Values

In [ ]:
missing = df.isnull().sum()

print("Columns with missing values: ", (missing > 0).sum())
print("Completely empty columns: ", (missing == len(df)).sum())

missing.sort_values(ascending = False).head(20)

df["F3924"].value_counts(normalize = True) * 100

missing_precentage = missing / len(df) * 100
missing_precentage.sort_values(ascending = False).head(100) 

In [ ]:
important_features = [
    "F115","F321","F527","F531","F670",
    "F1692","F2082","F2122","F2582",
    "F2678","F2737","F2956","F3043",
    "F3836","F3887","F3889","F3891","F3894"
]

df[important_features].info()
missing_precentage[important_features]


In [ ]:
df["F3891"].value_counts().head(20)
#pd.crosstab(df["F3891"], df["F3924"])
#pd.crosstab(df["F3891"], df["F3924"], normalize="index") * 100
pd.crosstab(df["F3891"], df["F3924"], normalize="columns") * 100

## Initial Fraud Correlation Analysis

In [ ]:
target_corr = df.corr(numeric_only=True)["F3924"].sort_values(ascending=False)

target_corr.head(20)

In [ ]:
df["F3912"].describe()
df["F3912"].value_counts()
pd.crosstab(df["F3912"], df["F3924"])
important_features = [
    "F115","F321","F527","F531","F670",
    "F1692","F2082","F2122","F2582",
    "F2678","F2737","F2956","F3043",
    "F3836","F3887","F3894"
]
target_corr.loc[important_features]

In [ ]:
(df["F2506"] == df["F2507"]).all()
pd.set_option("display.precision", 15)

target_corr.loc[["F2506", "F2507", "F2408", "F2409"]]

## Feature Redundancy Investigation

In [ ]:
df[["F2506", "F2507"]].corr()

In [ ]:
corr_matrix = df.corr(numeric_only=True)

In [ ]:
high_corr_count = (corr_matrix.abs() > 0.99).sum()

independent_features = high_corr_count[high_corr_count == 1]

len(independent_features)

independent_features.index[:100]

In [ ]:
df["F670"].value_counts()
pd.crosstab(df["F670"], df["F3924"])

In [ ]:
#df[["F2506","F2507"]].notna().all(axis=1).sum()
#pd.crosstab(df["F2506"], df["F3924"])
#df[["F2408","F2409"]].notna().all(axis=1).sum()
df[["F81","F82"]].notna().all(axis=1).sum()

Major finding:

Many extremely high correlations (>0.99) are calculated on very small overlapping subsets of data (e.g. 79 and 144 rows).

Therefore, correlation-based redundancy analysis is unreliable without considering overlap size.

Future analysis should incorporate both correlation strength and sample overlap.

In [ ]:
missing_indication = df.isnull().mean() * 100

sparse_cols = missing_indication[missing_indication > 99.5]

sparse_cols.head(20)

In [ ]:
sparse_cols = missing[missing > 99.5].index.tolist()

for col in sparse_cols[:50]:
    present_frauds = df.loc[df[col].notna(), "F3924"].sum()
    present_count = df[col].notna().sum()

    print(col, present_count, present_frauds)

In [ ]:
#(df["F1"].notna() == df["F4"].notna()).all()
#(df["F19"].notna() == df["F22"].notna()).all()
(df["F20"].notna() == df["F23"].notna()).all()

Major Finding:

Multiple feature groups share identical availability patterns.

Examples:
F1/F4
F2/F5
F3/F6
F19/F22/F25
F20/F23/F26
F21/F24/F27

This suggests features are generated in structured blocks,
possibly corresponding to banking products, customer segments,
or activity categories.

Missingness appears to be informative rather than random.

In [ ]:
for i in range(1, 50):
    print(i, df[f"F{i}"].notna().sum())

In [ ]:
(df["F1"].notna() == df["F7"].notna()).all()

In [ ]:
blockA = df["F1"].notna()
blockB = df["F7"].notna()

pd.crosstab(blockA, blockB)

Feature blocks exhibit structured overlap.

Example:
F1 block and F7 block are not identical populations.

However, 390 customers possess both feature groups.

This suggests missingness patterns may represent overlapping customer/product segments rather than random absence of data.

Presence indicators may therefore carry significant predictive information.

In [ ]:
block = df[df["F1"].notna()]

block[["F1","F2","F3","F4","F5","F6"]].describe()

In [ ]:
block = df[df["F1"].notna()]

block[["F1","F2","F3","F4","F5","F6"]].corr()

Strong evidence of templated feature generation.

F1-F6 form two mirrored feature groups.

Evidence:
- Identical missingness patterns
- Nearly identical distributions
- Extremely high pairwise correlations:
    F1-F4 = 0.964
    F2-F5 = 0.978
    F3-F6 = 0.982

Within-group relationships differ significantly,
suggesting corresponding metrics rather than duplicate features.

## Structured Feature Blocks

In [ ]:
block = df[df["F19"].notna()]

block[["F19","F20","F21","F22","F23","F24","F25","F26","F27"]].describe()

In [ ]:
block[["F19","F20","F21","F22","F23","F24","F25","F26","F27"]].corr()

In [ ]:
(df["F19"] == df["F25"]).all()
#(df["F20"] == df["F26"]).all()
#(df["F21"] == df["F27"]).all()

Major Finding:

Multiple feature blocks exhibit templated structure.

Observed characteristics:

1. Shared availability patterns.
2. Nearly identical distributions.
3. Extremely high correlations.
4. Not exact duplicates.

Conclusion:

These features likely represent transformed versions of the same underlying behavioral signal rather than duplicated data.

In [ ]:
(df["F19"] - df["F25"]).describe()

In [ ]:
df["F19"].equals(df["F25"])
df["F20"].equals(df["F26"])
df["F21"].equals(df["F27"])

## Missingness as a Signal

In [ ]:
df["F1"].equals(df["F4"])
df["F2"].equals(df["F5"])
df["F3"].equals(df["F6"])

In [ ]:
(F1 := df["F1"])
(F4 := df["F4"])

(F1 - F4).describe()

In [ ]:
mask = df["F1"].notna() & df["F4"].notna()

(df.loc[mask, "F1"] != df.loc[mask, "F4"]).sum()

In [ ]:
mask = df["F1"].notna() & df["F4"].notna()

pd.crosstab(
    (df.loc[mask, "F1"] != df.loc[mask, "F4"]),
    df.loc[mask, "F3924"]
)

In [ ]:
for i in range(660, 681):
    print(i, df[f"F{i}"].notna().sum())

In [ ]:
(df["F660"].notna() == df["F670"].notna()).all()

In [ ]:
df[["F660","F661","F662","F663","F664"]].describe()

In [ ]:
df[["F660","F661","F662","F663","F664"]].corr()

In [ ]:
for i in range(660, 681):
    print(f"F{i}", df[f"F{i}"].nunique(dropna=False))

In [ ]:
for col in ["F664","F665","F666","F670","F671","F672"]:
    print("\n", col)
    print(df[col].value_counts(dropna=False).head(10))

In [ ]:
for col in ["F664","F665","F666","F670"]:
    print("\n", col)
    print(pd.crosstab(df[col], df["F3924"]))

In [ ]:
for col in ["F667","F668","F669","F670","F671","F672","F673","F674","F675"]:
    print("\n", col)
    print(pd.crosstab(df[col], df["F3924"]))

In [ ]:
df.loc[df["F3924"] == 1, "F667"].describe()

In [ ]:
for col in ["F668","F669","F673","F674","F675"]:
    print(col)
    print(df.loc[df["F3924"] == 1, col].describe())

In [ ]:
#(df["F667"] > 0).sum()
(df["F668"] > 0).sum()
#(df["F669"] > 0).sum()

In [ ]:
(df["F664"] == (df["F667"] > 0).astype(int)).all()

In [ ]:
pd.crosstab(
    df["F664"],
    (df["F667"] > 0).astype(int)
)

In [ ]:
pd.crosstab(df["F665"], (df["F668"] > 0).astype(int))

In [ ]:
pd.crosstab(df["F666"], (df["F669"] > 0).astype(int))

In [ ]:
#pd.crosstab(df["F670"], (df["F673"] > 0).astype(int))
#pd.crosstab(df["F671"], (df["F674"] > 0).astype(int))
pd.crosstab(df["F672"], (df["F675"] > 0).astype(int))

Confirmed Feature Template

(F664, F667)
(F665, F668)
(F666, F669)

(F670, F673)
(F671, F674)
(F672, F675)

Relationship:
Flag == 1  <=>  Associated quantity > 0

Observed with perfect separation in all tested pairs.

In [ ]:
df.groupby("F670")["F673"].describe()

In [ ]:
subset = df[df["F670"] == 1]

subset.groupby("F3924")["F673"].describe()

In [ ]:
subset = df[df["F670"] == 1]

pd.crosstab(
    pd.qcut(subset["F673"], q=5, duplicates="drop"),
    subset["F3924"]
)

In [ ]:
subset = df[df["F671"] == 1]

pd.crosstab(
    pd.qcut(subset["F674"], q=5, duplicates="drop"),
    subset["F3924"]
)

In [ ]:
subset = df[df["F672"] == 1]

pd.crosstab(
    pd.qcut(subset["F675"], q=5, duplicates="drop"),
    subset["F3924"]
)

In [ ]:
features = [
    # Organizer features
    "F115", "F321", "F527", "F531",
     "F1692", "F2082", "F2122",
    "F2582", "F2678", "F2737", "F2956",
    "F3043", "F3836", "F3887", "F3894",

    # Customer features
    "F3889",  # behavior bucket
    "F3891",  # occupation

    # Feature family we discovered
    "F664", "F665", "F666",
    "F667", "F668", "F669",

    "F670", "F671", "F672",
    "F673", "F674", "F675"
]

target = "F3924"

family_cols = [
    "F664","F665","F666",
    "F667","F668","F669",
    "F670","F671","F672",
    "F673","F674","F675"
]

for col in family_cols:
    df[f"{col}_missing"] = df[col].isna().astype(int)

features += [f"{col}_missing" for col in family_cols]

X = df[features].copy()
y = df[target]

X = pd.get_dummies(
    X,
    columns=["F3889", "F3891"],
    dummy_na=True
)

X = X.fillna(-999)

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

In [ ]:
from sklearn.ensemble import RandomForestClassifier



rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)

In [ ]:

y_pred = rf.predict(X_test)

In [ ]:
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

import pandas as pd

importance = pd.Series(
    rf.feature_importances_,
    index=X.columns
)

importance = importance.sort_values(
    ascending=False
)

print(importance.head(30))


In [ ]:
from sklearn.metrics import confusion_matrix
print(confusion_matrix(y_test, y_pred))

In [ ]:
df["F2956"].value_counts()

In [ ]:
df.groupby("F3924")["F2956"].describe()

In [ ]:
for col in ["F2582", "F115", "F531", "F2737"]:
    print("\n", col)
    print(df.groupby("F3924")[col].describe())

In [ ]:
pd.crosstab(
    pd.qcut(df["F115"], q=10, duplicates="drop"),
    df["F3924"]
)

F115 Investigation

- Top 3 RF importance feature
- Fraud mean > Legit mean
- Fraud median > Legit median
- Highest F115 bucket contains 37/81 frauds
- Fraud rate in highest bucket ≈ 2.1%
- Overall fraud rate ≈ 0.9%

Conclusion:
Higher F115 values are associated with increased fraud risk.

In [ ]:
pd.crosstab(
    pd.qcut(df["F2956"], q=10, duplicates="drop"),
    df["F3924"]
)

#df["F2956"].value_counts()


In [ ]:
df["F2956_band"] = pd.cut(
    df["F2956"],
    bins=[-1,10,100,100000],
    labels=["Low","Mid","High"]
)

pd.crosstab(
    df["F2956_band"],
    df["F3924"]
)

df["F2956_mid"] = (
    (df["F2956"] > 10) &
    (df["F2956"] <= 100)
).astype(int)

In [ ]:
pd.crosstab(
    pd.qcut(df["F2582"], q=10, duplicates="drop"),
    df["F3924"]
)
#df["F531"].notnull().sum()

In [ ]:
frauds = df[df["F3924"] == 1]

f2582_hot = (
    (frauds["F2582"] > -0.03) &
    (frauds["F2582"] <= 0.0)
)

f2956_hot = (
    (frauds["F2956"] > 19) &
    (frauds["F2956"] <= 32)
)

print("F2582 hotspot:", f2582_hot.sum())
print("F2956 hotspot:", f2956_hot.sum())
print("Both:", (f2582_hot & f2956_hot).sum())

In [ ]:
f2582_hot = (
    (df["F2582"] > -0.03) &
    (df["F2582"] <= 0.0)
)

f2956_hot = (
    (df["F2956"] > 19) &
    (df["F2956"] <= 32)
)

group = pd.DataFrame({
    "F2582": f2582_hot,
    "F2956": f2956_hot
})

pd.crosstab(
    [group["F2582"], group["F2956"]],
    df["F3924"]
)

In [ ]:
df["fraud_cluster_1"] = (
    ((df["F2582"] > -0.03) & (df["F2582"] <= 0.0))
    &
    ((df["F2956"] > 19) & (df["F2956"] <= 32))
).astype(int)

In [ ]:
pd.crosstab(
    pd.qcut(df["F531"], q=10, duplicates="drop"),
    df["F3924"]
)

In [ ]:
ff2582_hot = (
    (df["F2582"] > -0.03) &
    (df["F2582"] <= 0.0)
)

f2956_hot = (
    (df["F2956"] > 19) &
    (df["F2956"] <= 32)
)

f531_hot = (
    (df["F531"] > 0.95) &
    (df["F531"] <= 1.35)
)

pd.crosstab(
    [f531_hot, f2582_hot, f2956_hot],
    df["F3924"]
)

In [ ]:
pd.crosstab(
    pd.qcut(
        df["F2737"],
        q=10,
        duplicates="drop"
    ),
    df["F3924"]
)

In [ ]:
f2737_safe = (
    (df["F2737"] > 0.0) &
    (df["F2737"] <= 0.04)
    
)
pd.crosstab(
    [f2737_safe],
    df["F3924"]
)

In [ ]:
safe2737 = (
    (df["F2737"] > 0.0) &
    (df["F2737"] <= 0.04)
)

pd.crosstab(
    [safe2737, f2582_hot],
    df["F3924"]
)
pd.crosstab(
    [safe2737, f2956_hot],
    df["F3924"]
)

In [ ]:
pd.crosstab(
    pd.qcut(
        df["F3836"],
        q=10,
        duplicates="drop"
    ),
    df["F3924"]
)

In [ ]:
f3836_hot = (
    (df["F3836"] > 148.596 ) &
    (df["F3836"] <= 20077.212)
)

pd.crosstab(
    [f3836_hot, f2956_hot, f2582_hot],
    df["F3924"]
)


In [ ]:
for col in ["F1","F2","F3","F4","F5","F6","F7"]:
    print("\n", col)
    print(df.groupby("F3924")[col].describe()[["mean","50%"]])

In [ ]:
from sklearn.ensemble import RandomForestClassifier

cols = [
    "F1","F2","F3",
    "F4","F5","F6",
    "F7","F8","F9",
    "F10","F11","F12"
]


X = df[cols].copy()
y = df[target]
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

rf = RandomForestClassifier(
    n_estimators=300,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1
)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)
from sklearn.metrics import classification_report

print(classification_report(y_test, y_pred))

import pandas as pd

importance = pd.Series(
    rf.feature_importances_,
    index=X.columns
)

importance = importance.sort_values(
    ascending=False
)

print(importance.head(30))


In [ ]:
pd.crosstab(
    pd.qcut(df["F7"], q=10, duplicates="drop"),
    df["F3924"]
)

In [ ]:
df["F1_family_mean"] = df[
    ["F1","F2","F3","F4","F5","F6","F7"]
].mean(axis=1)

In [ ]:
pd.crosstab(
    pd.qcut(
        df["F1_family_mean"],
        q=10,
        duplicates="drop"
    ),
    df["F3924"]
)

In [ ]:
pd.crosstab(
    pd.qcut(
        df["F5"],
        q=10,
        duplicates="drop"
    ),
    df["F3924"]
)

In [ ]:
df.groupby("F3924")["F2122"].describe()

In [ ]:
pd.crosstab(
    pd.qcut(df["F2122"], 10, duplicates="drop"),
    df["F3924"]
)

In [ ]:
df.groupby("F3924")["F527"].describe()

In [ ]:
pd.crosstab(
    pd.qcut(df["F527"], 10, duplicates="drop"),
    df["F3924"]
)

In [ ]:
df.groupby("F3924")["F321"].describe()

In [ ]:
pd.crosstab(
    pd.qcut(df["F321"], 10, duplicates="drop"),
    df["F3924"]
)

In [ ]:
df.groupby("F3924")["F3894"].describe()

In [ ]:
pd.crosstab(
    pd.qcut(df["F3894"], 10, duplicates="drop"),
    df["F3924"]
)

In [ ]:
df.groupby("F3924")["F2737"].describe()

In [ ]:
pd.crosstab(
    pd.qcut(df["F2737"], 10, duplicates="drop"),
    df["F3924"]
)

In [ ]:
df[["F3836","F2122","F527","F3894","F321","F2956", "F1","F2","F3","F4","F5","F6","F7","F8","F9","F10","F12"]].corr()

In [ ]:
for col in [
    "F1_missing",
    "F2_missing",
    "F3_missing",
    "F4_missing"
]:
    print(col)
    print(pd.crosstab(df[col], df["F3924"]))
    print()

In [ ]:
pd.crosstab(df["F3891"], df["F3924"])

In [ ]:
frauds = df[df["F3924"] == 1]

frauds[
    frauds["F2582"] > 0.15
][["F2582","F531","F115","F2956"]]

In [ ]:
legits = df[df["F3924"] == 0]

(
    (legits["F2582"] > 0.15)
).mean()

In [ ]:
pd.crosstab(
    df["F2582"] > 0.15,
    df["F3924"]
)

In [ ]:
positive_f2582_frauds = df[
    (df["F3924"] == 1) &
    (df["F2582"] > 0.15)
]

print(len(positive_f2582_frauds))

positive_f2582_frauds[
    ["F115","F2582","F2956","F531","F2737"]
]

In [ ]:
positive_region = df[df["F2582"] > 0.15]

positive_region.groupby("F3924")[[
    "F115",
    "F2956",
    "F531",
    "F2737"
]].median()

In [ ]:
positive_region.groupby("F3924")[[
    "F115",
    "F2956",
    "F531",
    "F2737"
]].mean()

In [ ]:
positive_region["F2956_bin"] = pd.cut(
    positive_region["F2956"],
    bins=[0,20,40,60,80,120,10000]
)

pd.crosstab(
    positive_region["F2956_bin"],
    positive_region["F3924"]
)

In [ ]:
df["positive2582_low2956"] = (
    (df["F2582"] > 0.15) &
    (df["F2956"] < 60)
).astype(int)

In [ ]:
pd.crosstab(
    df["positive2582_low2956"],
    df["F3924"]
)

In [ ]:
df["F2737_gt2"] = (df["F2737"] > 2).astype(int)

pd.crosstab(
    df["F2737_gt2"],
    df["F3924"]
)

In [ ]:
df[df["F2737"] > 2][
    ["F3924","F115","F2582","F2956","F531","F2737"]
].sort_values("F3924", ascending=False)